# Optimization for 3D Scene Reconstruction

**A Comparative Study of Optimizers, Losses, and Representations for NeRF and 3D Gaussian Splatting**

**Group:** António Cruz (140129), Duarte Cabrita (120058)

**Course:** Computational Optimization — Phase 2

**Submission date:** 11 June 2026

---


# 1. Introduction

This notebook is the Phase 2 deliverable of the Computational Optimization project. It studies computational optimization through an applied problem in computer vision: reconstructing a 3D scene from a sparse set of 2D photographs so that new, never-photographed viewpoints can be synthesised.

The reconstruction is framed as a continuous, non-convex, stochastic optimization problem and solved with a Neural Radiance Field (NeRF): a small multi-layer perceptron, paired with a differentiable volume renderer, whose weights are fitted by gradient descent. The project is deliberately comparative. Its contributions are:

- **Five optimizers implemented from scratch** on top of PyTorch autograd — SGD, SGD with classical momentum, SGD with Nesterov accelerated gradient, Adam, and AdamW — plus a cosine-annealing learning-rate schedule. No `torch.optim` method is used for any compared run.
- **Four loss formulations** — L2, L1, SSIM, and a weighted L1+SSIM combination.
- **A controlled experiment harness** that runs every method under identical data, initialisation, seed, and compute budgets, and logs convergence, stability, and final reconstruction quality.
- **A 3D Gaussian Splatting baseline**, contrasted with NeRF on the same scenes and metrics, to place the optimizer study against an alternative scene representation.
- **Candidate improvements** motivated by the bottleneck the comparison reveals.

Reconstruction quality is measured with PSNR, SSIM, and LPIPS on held-out views. The configuration that all experiments share — rendering resolution, MLP size, and iteration budget — was fixed empirically in a scoping study reported in Section 6.

The notebook is organised so that the model and renderer (Section 3), the optimizers (Section 4), and the loss functions (Section 5) are defined first; the experimental setup and harness follow (Section 6); and the comparison experiments and their analysis make up Sections 7 to 12.


In [ ]:
# Environment: imports, device, and reproducibility.
# The LPIPS metric needs pretrained backbone weights; they are cached under
# data/models, so TORCH_HOME is pointed there before lpips is ever imported.
import os
os.environ["TORCH_HOME"] = os.path.abspath("../data/models")

import math
import json
import time
import random
import hashlib
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
from skimage.metrics import structural_similarity as ssim_metric

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def seed_everything(seed):
    """Fix every random source so a run is repeatable across sessions."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(0)
print(f"Device: {DEVICE} | PyTorch {torch.__version__}")


# 2. Problem Formulation

## The task

3D scene reconstruction from a sparse set of 2D photographs, framed as a continuous non-convex optimization problem.

## Mathematical formulation

Let $\theta$ be the parameters (weights and biases) of a small multi-layer perceptron $f_\theta$ that maps a 3D point $(x, y, z)$ to a density $\sigma \ge 0$ and an RGB colour $c \in [0, 1]^3$.

Let $R(\theta; \pi)$ be the differentiable volume-rendering operator: given $\theta$ and a camera pose $\pi$, it casts rays through every pixel, samples points along each ray, queries $f_\theta$, and composites the samples into a synthesised image.

Given captured images and their camera poses $\{(I_i, \pi_i)\}_{i=1..N}$, the reconstruction minimises the average discrepancy between rendered and observed pixels:

$$\min_\theta \ \ \frac{1}{N}\sum_{i=1}^{N} \mathcal{L}\big( R(\theta; \pi_i),\, I_i \big)$$

where $\mathcal{L}$ is one of the loss formulations of Section 5 (the baseline being the squared error $\lVert \cdot \rVert_2^2$).

The problem is non-convex (the rendering operator composes with the MLP non-linearities) and high-dimensional ($|\theta|$ on the order of $10^5$ parameters even for the small model used here). It is solved by stochastic gradient methods: at each iteration a random image and a random batch of its pixel rays are drawn, rendered, scored against the ground truth, and used to update $\theta$.

## What this project compares

The formulation above leaves three components open, and the project varies each in a controlled way:

- **The optimizer** that performs the update $\theta \leftarrow \theta - \dots$ (Section 4).
- **The loss** $\mathcal{L}$ that defines the objective surface (Section 5).
- **The scene representation** itself — NeRF versus 3D Gaussian Splatting (Section 10).

Convergence speed, training stability, and final reconstruction quality are measured for each, under identical conditions, by the harness of Section 6.

---


# 3. NeRF: Model and Differentiable Renderer

This section defines the scene representation and the rendering pipeline that together realise $f_\theta$ and the rendering operator $R(\theta;\pi)$ of the formulation. The components are: loading a scene's images and camera poses (3.1), generating one camera ray per pixel (3.2), lifting 3D coordinates into a high-frequency feature space (3.3), the MLP that predicts density and colour (3.4), and the volume renderer that composites those predictions into pixels (3.5).


## 3.1 Data Loading

The experiments use the `nerf_synthetic` dataset: eight synthetic scenes, each rendered from many known camera poses. Every scene ships with three pose sets — `transforms_train.json`, `transforms_val.json`, and `transforms_test.json` — which the project uses directly as the train / validation / test split (Section 6.2).

The loader below reads one split of one scene. Each image is RGBA: the loader composites it over a white background (so the transparent surround becomes white rather than black) and area-downsamples it from the native 800x800 to the working resolution. The focal length is derived from the scene's horizontal field of view and rescaled to the working resolution. This function is the only place the dataset format is parsed; everything downstream consumes plain tensors.


In [ ]:
DATA_ROOT = "../data/nerf_synthetic"


def load_scene(scene, split, resolution, n_views=None):
    """Load one split of one nerf_synthetic scene.

    Returns
        images : float tensor [V, resolution, resolution, 3], RGB in [0, 1]
        poses  : float tensor [V, 4, 4], camera-to-world matrices
        focal  : float, focal length in pixels at the working resolution
    """
    scene_dir = os.path.join(DATA_ROOT, scene)
    with open(os.path.join(scene_dir, f"transforms_{split}.json")) as f:
        meta = json.load(f)
    frames = meta["frames"]
    if n_views is not None:
        frames = frames[:n_views]

    imgs, poses = [], []
    for fr in frames:
        path = os.path.join(scene_dir, fr["file_path"] + ".png")
        rgba = np.asarray(Image.open(path), dtype=np.float32) / 255.0
        # Composite RGBA over a white background.
        rgb = rgba[..., :3] * rgba[..., 3:4] + (1.0 - rgba[..., 3:4])
        imgs.append(rgb)
        poses.append(np.asarray(fr["transform_matrix"], dtype=np.float32))

    images = torch.from_numpy(np.stack(imgs))          # [V, 800, 800, 3]
    poses = torch.from_numpy(np.stack(poses))          # [V, 4, 4]
    native = images.shape[1]

    # Area-downsample to the working resolution.
    x = images.permute(0, 3, 1, 2)
    x = F.interpolate(x, size=(resolution, resolution), mode="area")
    images = x.permute(0, 2, 3, 1).contiguous()

    focal = 0.5 * native / math.tan(0.5 * float(meta["camera_angle_x"]))
    focal = focal * resolution / native
    return images, poses, float(focal)


## 3.2 Ray Generation

For a given image resolution and camera pose, generate a ray (origin and direction in world coordinates) for every pixel. These rays are the inputs to the volume-rendering integral. The rendering operator $R(\theta;\pi)$ of the formulation is realised by this function together with the volume renderer of Section 3.5.


In [ ]:
def get_rays(H, W, focal, pose):
    """Return ray origins and directions for every pixel in an HxW image."""
    i, j = torch.meshgrid(
        torch.arange(W, device=DEVICE).float(),
        torch.arange(H, device=DEVICE).float(),
        indexing="xy",
    )
    dirs = torch.stack([(i - W * 0.5) / focal, -(j - H * 0.5) / focal, -torch.ones_like(i)], dim=-1)
    rays_d = (dirs @ pose[:3, :3].T)
    rays_o = pose[:3, 3].expand(rays_d.shape)
    return rays_o, rays_d


## 3.3 Positional Encoding

Map raw 3D coordinates into a higher-dimensional space using sines and cosines at exponentially increasing frequencies. Without this lifting, a small MLP cannot fit high-frequency scene detail. This is a fixed (non-learned) feature map; only the MLP weights are optimization variables.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, num_freqs=10):
        super().__init__()
        self.freqs = 2.0 ** torch.arange(num_freqs).float().to(DEVICE)
        self.out_dim = 3 + 3 * 2 * num_freqs

    def forward(self, x):
        encs = [x]
        for f in self.freqs:
            encs += [torch.sin(f * x), torch.cos(f * x)]
        return torch.cat(encs, dim=-1)


## 3.4 NeRF MLP

The optimization variable: a small fully-connected network mapping the encoded 3D point to a density and an RGB colour. A ReLU on the density head guarantees non-negativity; a sigmoid on the colour head bounds it to $[0, 1]^3$.

The architecture — four layers of width 128, roughly 58,000 parameters — was selected by the Stage 0 scoping study (Section 6.1): a far larger network raised reconstruction quality by only a few tenths of a dB for several times the compute, which is not a worthwhile trade in a study whose subject is the optimizer rather than the absolute reconstruction quality.


In [ ]:
class TinyNeRF(nn.Module):
    def __init__(self, enc_dim, width=128, depth=4):
        super().__init__()
        layers = [nn.Linear(enc_dim, width), nn.ReLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.ReLU()]
        self.trunk = nn.Sequential(*layers)
        self.density = nn.Linear(width, 1)
        self.rgb = nn.Linear(width, 3)

    def forward(self, x):
        h = self.trunk(x)
        sigma = torch.relu(self.density(h))[..., 0]
        c = torch.sigmoid(self.rgb(h))
        return sigma, c


## 3.5 Volume Rendering

Composite the MLP's per-point predictions into per-pixel colours via the discretised volume-rendering integral. For each ray we sample $N$ points between the near and far planes, query density and colour, and accumulate them using the alpha-compositing weights derived from accumulated transmittance. This is the differentiable rendering operator $R(\theta;\pi)$, and it is differentiable end to end, so gradients of the image loss flow back into the MLP weights.


In [ ]:
def render_rays(rays_o, rays_d, model, encoding, near=2.0, far=6.0, N=64):
    t = torch.linspace(near, far, N, device=DEVICE)
    delta = (far - near) / N
    t = t + (torch.rand(rays_o.shape[0], N, device=DEVICE) - 0.5) * delta
    pts = rays_o[:, None] + rays_d[:, None] * t[:, :, None]
    sigma, c = model(encoding(pts.reshape(-1, 3)))
    sigma = sigma.reshape(rays_o.shape[0], N)
    c = c.reshape(rays_o.shape[0], N, 3)
    deltas = torch.cat([t[:, 1:] - t[:, :-1], torch.full_like(t[:, :1], 1e10)], dim=-1)
    alpha = 1.0 - torch.exp(-sigma * deltas)
    T = torch.cumprod(torch.cat([torch.ones_like(alpha[:, :1]), 1 - alpha + 1e-10], dim=-1), dim=-1)[:, :-1]
    w = T * alpha
    rgb = (w[..., None] * c).sum(dim=1)
    return rgb


### Rendering a full image

`render_rays` works on an arbitrary batch of rays. Evaluation and qualitative previews need a complete image, so the helper below renders every pixel of an `H x W` view, in ray chunks to bound memory. It runs under `torch.inference_mode`, since no gradient is needed when rendering for evaluation.


In [ ]:
@torch.inference_mode()
def render_full(model, encoding, pose, H, W, focal, chunk=8192,
                near=2.0, far=6.0, n_samples=64):
    """Render a complete H x W image from one camera pose, in ray chunks."""
    rays_o, rays_d = get_rays(H, W, focal, pose)
    rays_o, rays_d = rays_o.reshape(-1, 3), rays_d.reshape(-1, 3)
    out = [render_rays(rays_o[i:i + chunk], rays_d[i:i + chunk], model, encoding,
                       near=near, far=far, N=n_samples)
           for i in range(0, rays_o.shape[0], chunk)]
    return torch.cat(out).reshape(H, W, 3)


# 4. Optimizers

This project implements five gradient-based optimizers from scratch, on top of PyTorch's automatic differentiation. Implementing them ourselves rather than calling `torch.optim` is what makes the comparison a genuine study of computational optimization, and it guarantees every method runs under identical numerical conventions.

All optimizers expose the same minimal interface, `zero_grad()` and `step()`, so the training loop can use any of them interchangeably. Plain SGD, momentum, and Nesterov are the three modes of a single `MySGD` class; `MyAdam` and `MyAdamW` are separate. A cosine-annealing learning-rate schedule with linear warmup (Section 4.6) can be applied on top of any of them.


## 4.1 Stochastic Gradient Descent

Plain SGD is the reference baseline: each parameter moves directly down its gradient, $\theta \leftarrow \theta - \eta\, g$, with a single global step size $\eta$. It has no state and no per-parameter scaling, so it is the most exposed to ill-conditioning — directions of high curvature force a small $\eta$, which then makes progress along low-curvature directions slow.

The `MySGD` class below implements plain SGD together with its momentum and Nesterov variants (Sections 4.2 and 4.3) as modes of one class, selected by its constructor arguments, so all three share a single tested code path.


In [ ]:
class MySGD:
    """SGD with optional momentum and Nesterov acceleration.
    momentum=0             -> plain SGD:  theta <- theta - lr*g
    momentum>0, nesterov=F -> classical:  v <- mu*v + g;  theta <- theta - lr*v
    momentum>0, nesterov=T -> Nesterov:   v <- mu*v + g;  theta <- theta - lr*(g + mu*v)
    """
    def __init__(self, params, lr=1e-3, momentum=0.0, nesterov=False):
        self.params = [p for p in params if p.requires_grad]
        self.lr, self.mu, self.nesterov = lr, momentum, nesterov
        self.v = [torch.zeros_like(p) for p in self.params] if momentum > 0 else None

    @torch.no_grad()
    def step(self):
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            g = p.grad
            if self.mu > 0:
                v = self.v[i]
                v.mul_(self.mu).add_(g)
                p.add_(g + self.mu * v if self.nesterov else v, alpha=-self.lr)
            else:
                p.add_(g, alpha=-self.lr)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_(); p.grad.zero_()


## 4.2 SGD with Classical Momentum

Momentum accumulates an exponentially-weighted running sum of past gradients, the velocity $v \leftarrow \mu v + g$, and steps along it: $\theta \leftarrow \theta - \eta\, v$. The decay $\mu$ (here $0.9$) damps the oscillation that plain SGD suffers across high-curvature directions, while letting consistent descent directions build up speed. This usually gives faster and steadier convergence. It is the `momentum > 0`, `nesterov=False` mode of `MySGD` defined above.


## 4.3 SGD with Nesterov Accelerated Gradient

Nesterov's accelerated gradient evaluates the descent direction with a look-ahead: the velocity is updated as in classical momentum, but the step applied is $\theta \leftarrow \theta - \eta\,(g + \mu v)$, which corresponds to measuring the gradient slightly ahead, where the momentum term is about to carry the parameters. The correction reduces overshoot near the minimum and, on well-behaved objectives, improves the convergence rate. It is the `nesterov=True` mode of `MySGD`.


## 4.4 Adam

Adam keeps two exponentially-weighted moment estimates per parameter: the first moment $m$ (a smoothed gradient, like momentum) and the second moment $v$ (a smoothed squared gradient). The update divides the first moment by the square root of the second, $\theta \leftarrow \theta - \eta\, \hat m / (\sqrt{\hat v} + \epsilon)$, giving every parameter its own effective step size — large where gradients are small and consistent, small where they are large or noisy. Both estimates start at zero and are therefore bias-corrected ($\hat m, \hat v$) so the early steps are not damped. This per-parameter adaptation is what lets Adam cope with the ill-conditioning that slows plain SGD.


In [ ]:
class MyAdam:
    """Adam (Kingma & Ba, 2014)."""
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = [p for p in params if p.requires_grad]
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    @torch.no_grad()
    def step(self):
        self.t += 1
        bc1 = 1 - self.b1 ** self.t
        bc2 = 1 - self.b2 ** self.t
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            g = p.grad
            m.mul_(self.b1).add_(g, alpha=1 - self.b1)
            v.mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
            p.addcdiv_(m / bc1, v.div(bc2).sqrt().add_(self.eps), value=-self.lr)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_(); p.grad.zero_()


## 4.5 AdamW

AdamW is Adam with *decoupled* weight decay. Standard L2 regularisation adds $\lambda\theta$ to the gradient, so it passes through Adam's per-parameter scaling and is effectively rescaled differently for every weight. AdamW instead applies the decay straight to the parameter, $\theta \leftarrow \theta - \eta\lambda\,\theta$, separately from the adaptive gradient step. The regularisation strength is then uniform and independent of the gradient statistics, which is the behaviour usually intended by "weight decay".


In [ ]:
class MyAdamW:
    """AdamW (Loshchilov & Hutter, 2017): Adam with decoupled weight decay.
    The weight-decay term is applied directly to the parameter, separately
    from the adaptive gradient step."""
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=1e-2):
        self.params = [p for p in params if p.requires_grad]
        self.lr, self.b1, self.b2, self.eps, self.wd = lr, beta1, beta2, eps, weight_decay
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    @torch.no_grad()
    def step(self):
        self.t += 1
        bc1 = 1 - self.b1 ** self.t
        bc2 = 1 - self.b2 ** self.t
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            g = p.grad
            m.mul_(self.b1).add_(g, alpha=1 - self.b1)
            v.mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
            p.addcdiv_(m / bc1, v.div(bc2).sqrt().add_(self.eps), value=-self.lr)
            p.add_(p, alpha=-self.lr * self.wd)   # decoupled weight decay

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_(); p.grad.zero_()


## 4.6 Learning-Rate Schedule

All five optimizers above take a fixed base learning rate. The schedule below modulates it over training, independently of which optimizer is used. It combines a short *linear warmup* — the rate ramps up from zero over the first few hundred steps, avoiding a destructive early step while the moment estimates are still cold — with *cosine annealing*, which then eases the rate smoothly down to zero so the final iterations settle into the minimum instead of bouncing around it. The schedule is an optional factor applied on top of any optimizer, and its effect is one of the comparisons in Section 7.


In [ ]:
def cosine_warmup_lr(step, base_lr, warmup_steps, total_steps):
    """Cosine-annealing learning-rate schedule with linear warmup.
    Returns the learning rate to use at the given training step."""
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))


# 5. Loss Functions

The loss $\mathcal{L}$ in the formulation scores a rendered image region against the observed one, and so defines the surface the optimizer descends. The project compares four formulations:

- **L2** — mean squared error. The baseline; it corresponds directly to maximising PSNR. Smooth in its argument, but it averages errors and so tolerates a uniformly slightly-wrong, blurry reconstruction.
- **L1** — mean absolute error. Penalises large and small errors more evenly, is less swayed by outlier pixels, and often preserves edges better than L2.
- **SSIM** — structural similarity. A perceptual measure comparing local luminance, contrast, and structure rather than raw pixel differences; used as a loss in the form $1 - \mathrm{SSIM}$.
- **L1 + SSIM** — a weighted combination, $\alpha\,\mathrm{L1} + (1-\alpha)\,(1-\mathrm{SSIM})$, pairing L1's pixel accuracy with SSIM's structural sensitivity.

L2 and L1 are pixel-wise and are defined below; they are the losses used for the optimizer comparison of Section 7. SSIM is a *spatial* measure and requires rendering contiguous image patches rather than scattered pixels, so the SSIM-based losses, and the patch-sampling path they need, are introduced with the loss comparison in Section 8. Every loss exposes the same `(prediction, target) -> scalar` interface and is selected by name through the `make_loss` factory, which keeps the harness loss-agnostic.


In [ ]:
def loss_l2(pred, target):
    """Mean squared error (the PSNR-aligned baseline)."""
    return ((pred - target) ** 2).mean()


def loss_l1(pred, target):
    """Mean absolute error."""
    return (pred - target).abs().mean()


# Loss registry. SSIM and L1+SSIM are added in Section 8 (they need patch
# sampling); registering by name keeps the harness loss-agnostic.
LOSSES = {
    "l2": loss_l2,
    "l1": loss_l1,
}


def make_loss(name):
    """Return the loss callable registered under `name`."""
    if name not in LOSSES:
        raise KeyError(f"unknown loss '{name}'; available: {sorted(LOSSES)}")
    return LOSSES[name]


# 6. Experimental Setup

This section fixes what the comparison experiments hold constant and builds the machinery that runs them. Section 6.1 is the Stage 0 scoping study that chose the rendering resolution, the MLP size, and the iteration budget by measurement before the main experiment matrix was committed. Section 6.2 defines the train / validation / test split and the three evaluation metrics. Section 6.3 is the experiment harness: a single function that takes a run configuration, trains a NeRF under it, logs the full history to disk, and returns the result, so that the experiments of Sections 7 to 11 reduce to calls into it.


## 6.1 Stage 0: Configuration Selection


### Project context

This project studies computational optimization in the context of an AI application: 3D scene reconstruction from 2D photographs, framed as a continuous, non-convex, stochastic optimization problem. Its core is comparative — it implements five gradient-based optimizers from scratch, several loss formulations, and contrasts the NeRF representation against 3D Gaussian Splatting, analysing convergence, stability, and performance across all of them. That comparison is carried out as a large experiment matrix: on the order of a hundred training runs once optimizers, loss functions, random seeds, and scenes are crossed.

### Why this investigation exists

Two settings apply uniformly to every run in that matrix and so must be fixed before it begins: the rendering resolution and the MLP architecture. Together they determine the total compute the matrix consumes — whether it is feasible at all on the available hardware — and the quality regime in which the optimizer comparison operates, that is, whether the methods produce differences large enough to observe and reason about. Choosing them by assumption risks either an infeasible compute budget or a comparison conducted in an uninformative regime. This Stage 0 investigation therefore fixes both choices empirically, by measurement on a single representative scene (Lego), before the Phase 2 matrix is committed.

### Question

At what image resolution, and with what MLP architecture, should the Phase 2 experiments run? Concretely: (1) how do reconstruction cost and quality scale with rendering resolution; and (2) does enlarging the MLP lift the quality ceiling, particularly at high resolution, by enough to justify its additional compute cost?


### Fixed configuration

Held constant across every Stage 0 run:

- **Scene:** `nerf_synthetic` Lego — 100 training images; 3 held-out views for evaluation.
- **Optimizer:** custom-implemented Adam — learning rate `5e-4`, β1 `0.9`, β2 `0.999`, ε `1e-8`.
- **Loss:** L2 (mean squared error between rendered and observed RGB).
- **Volume rendering:** 64 samples per ray, stratified; near plane `2.0`, far plane `6.0`.
- **Positional encoding:** 10 frequency bands of sin/cos, giving an MLP input dimension of 63.
- **Output heads:** density via Linear→1 with ReLU; colour via Linear→3 with sigmoid.
- **Random seed:** `0`.
- **Data preparation:** RGBA composited over a white background, area-downsampled from native 800×800.

### MLP architectures compared

- **Small MLP** — 4 fully-connected layers of width 128, plain feedforward (Linear + ReLU), no skip connection. Approximately 58,000 parameters. The architecture used in the proof of concept.
- **Large MLP, no skip** — 8 layers of width 256, plain feedforward. This configuration collapsed during training (density → 0 everywhere, PSNR ≈ 1.2 dB): a plain 8-layer feedforward MLP is not trainable in this setup.
- **Large MLP, with skip** — 8 layers of width 256, with the 63-dimensional encoded input concatenated into the input of layer 4 (the standard NeRF skip connection). 494,084 parameters. Trains correctly.

The completed runs and their results are tabulated below.


In [ ]:
import pandas as pd

# Stage 0 measured results. Each row is one completed training run.
# (The failed plain-8-layer run is omitted; it collapsed to PSNR ~1.2 dB.)
stage0_results = pd.DataFrame([
    {"run": "1 resolution sweep",    "resolution": "100x100", "batch_rays": 1024, "iterations": 20000, "MLP": "small 128x4",      "best_PSNR_dB": 23.51, "best_SSIM": 0.890},
    {"run": "1 resolution sweep",    "resolution": "200x200", "batch_rays": 1024, "iterations": 20000, "MLP": "small 128x4",      "best_PSNR_dB": 22.35, "best_SSIM": 0.833},
    {"run": "1 resolution sweep",    "resolution": "400x400", "batch_rays": 1024, "iterations": 20000, "MLP": "small 128x4",      "best_PSNR_dB": 21.58, "best_SSIM": 0.788},
    {"run": "1 resolution sweep",    "resolution": "800x800", "batch_rays": 1024, "iterations": 20000, "MLP": "small 128x4",      "best_PSNR_dB": 21.15, "best_SSIM": 0.775},
    {"run": "2 batch/iter scale-up", "resolution": "200x200", "batch_rays": 4096, "iterations": 40000, "MLP": "small 128x4",      "best_PSNR_dB": 23.08, "best_SSIM": 0.856},
    {"run": "2 batch/iter scale-up", "resolution": "800x800", "batch_rays": 4096, "iterations": 40000, "MLP": "small 128x4",      "best_PSNR_dB": 21.68, "best_SSIM": 0.786},
    {"run": "4 larger MLP",          "resolution": "200x200", "batch_rays": 4096, "iterations": 40000, "MLP": "large 256x8 skip", "best_PSNR_dB": 23.41, "best_SSIM": 0.873},
    {"run": "4 larger MLP",          "resolution": "800x800", "batch_rays": 4096, "iterations": 40000, "MLP": "large 256x8 skip", "best_PSNR_dB": 22.07, "best_SSIM": 0.799},
])
stage0_results


### Findings

**Per-iteration training cost is independent of resolution.** Measured per-iteration time was essentially flat across 100/200/400/800 (~3.8 ms at batch 1024; ~11.5 ms at batch 4096). NeRF training samples a fixed batch of rays per step, so the gradient step does not depend on image size. Resolution is therefore not a per-step compute cost.

**Reconstruction quality decreases with resolution at a fixed budget.** With a fixed ray batch, a larger image is covered a smaller fraction per iteration: at 100×100 a 1024-ray batch is about 10% of an image, at 800×800 about 0.16%. Higher resolutions are consequently under-trained at a fixed iteration count, and best PSNR falls from ~23.5 dB at 100×100 to ~21.2 dB at 800×800.

**Evaluation cost scales quadratically with resolution.** Rendering a full held-out view is an H×W operation; measured full-frame render time rose from 0.011 s at 100×100 to 0.672 s at 800×800.

**Compute is not the binding constraint.** Estimated total compute for the roughly 100-run Phase 2 matrix is only a few hours at any tested resolution, and peak VRAM stayed under ~1.7 GB throughout.

**A larger MLP yields only a marginal quality gain.** The large MLP (256×8 with skip, 494k parameters) improved best PSNR by just +0.33 dB at 200×200 and +0.39 dB at 800×800 over the small MLP (~58k parameters), at roughly 3.7× the per-iteration cost. A plain 8-layer MLP without a skip connection failed to train at all. Neither more capacity nor (separately tested) 8× more ray coverage closed the gap between 800×800 and 200×200.


### Configuration selected for Phase 2

**MLP architecture — small MLP (width 128, depth 4).** The large MLP's ~0.35 dB average gain does not justify ~3.7× the per-iteration cost across a roughly 100-run matrix; absolute PSNR is not the objective of an optimizer-comparison study.

**Rendering resolution — 200×200.** A dedicated SGD-vs-Adam separability check confirmed that 200×200 distinguishes optimizers fully: the Adam−SGD best-PSNR gap was 12.77 dB at 200×200, marginally larger than the 12.00 dB measured at 400×400. Combined with 200×200's better quality-per-compute — it avoids the coverage penalty that lowers PSNR at higher resolution under a fixed ray budget — 200×200 is selected. The concern that the lower resolution might be an "uninformative regime" is refuted by measurement.

**Iteration budget — fixed 40,000-iteration anytime-performance budget.** Phase 2 measures which optimizer reaches the best state within a fixed compute budget, rather than letting the slowest optimizer dictate an open-ended one.


## 6.2 Dataset Splits and Evaluation Metrics

**Splits.** Each `nerf_synthetic` scene provides three disjoint pose sets, which the project uses directly:

- **Train** — the full `transforms_train` set (100 views). Mini-batches of rays are drawn from these during optimization.
- **Validation** — the first `N_VAL_VIEWS` poses of `transforms_val`. Rendered periodically during training to trace the convergence curves and to choose each optimizer's learning rate in the Section 7 sweep. Never trained on.
- **Test** — the first `N_TEST_VIEWS` poses of `transforms_test`. Rendered once, at the end of a run, to produce the reported quality metrics. Never trained on and never used for any selection, so the reported numbers are an honest held-out estimate.

**Metrics.** Reconstruction quality on the held-out views is measured three ways, because each captures a different notion of "correct":

- **PSNR** (dB, higher better) — a logarithmic transform of mean squared error; a pure pixel-fidelity measure.
- **SSIM** ($[0,1]$, higher better) — structural similarity; compares local luminance, contrast, and structure, so it rewards getting the *layout* of detail right.
- **LPIPS** ($[0,1]$, lower better) — a learned perceptual distance: the distance between deep features of a pretrained network. It tracks human judgements of similarity better than PSNR or SSIM, and penalises perceptual artefacts the other two miss.

PSNR and SSIM are cheap and computed from NumPy. LPIPS uses a pretrained AlexNet backbone, whose weights are cached under `data/models` (Section 1 sets `TORCH_HOME` accordingly) so no network access is needed. It is constructed lazily, on first use.


In [ ]:
N_VAL_VIEWS = 5      # validation views: convergence curves and LR selection
N_TEST_VIEWS = 10    # test views: final reported metrics only

# LPIPS is constructed once, on first use (it loads the cached AlexNet weights).
_lpips_model = None


def get_lpips():
    """Return the LPIPS metric, constructing it once on first call."""
    global _lpips_model
    if _lpips_model is None:
        import lpips
        _lpips_model = lpips.LPIPS(net="alex", verbose=False).to(DEVICE)
        for p in _lpips_model.parameters():
            p.requires_grad_(False)
    return _lpips_model


def psnr_from_mse(mse):
    """Peak signal-to-noise ratio in dB from a mean squared error."""
    return -10.0 * math.log10(max(mse, 1e-10))


@torch.inference_mode()
def evaluate(model, encoding, images, poses, focal,
             near=2.0, far=6.0, n_samples=64, with_lpips=False):
    """Render every pose and score it against the matching image.

    Returns the mean PSNR and SSIM over the views, and the mean LPIPS too
    when `with_lpips` is set."""
    H = W = images.shape[1]
    psnrs, ssims, lpips_vals = [], [], []
    lpips_fn = get_lpips() if with_lpips else None
    for gt, pose in zip(images, poses):
        pred = render_full(model, encoding, pose, H, W, focal,
                           near=near, far=far, n_samples=n_samples).clamp(0, 1)
        pred_np, gt_np = pred.cpu().numpy(), gt.cpu().numpy()
        psnrs.append(psnr_from_mse(float(((pred_np - gt_np) ** 2).mean())))
        ssims.append(float(ssim_metric(pred_np, gt_np,
                                       channel_axis=-1, data_range=1.0)))
        if lpips_fn is not None:
            # LPIPS expects [N, 3, H, W] tensors in [-1, 1].
            a = pred.permute(2, 0, 1)[None] * 2 - 1
            b = gt.to(DEVICE).permute(2, 0, 1)[None] * 2 - 1
            lpips_vals.append(float(lpips_fn(a, b).item()))
    out = {"psnr": float(np.mean(psnrs)), "ssim": float(np.mean(ssims))}
    if with_lpips:
        out["lpips"] = float(np.mean(lpips_vals))
    return out


## 6.3 Experiment Harness

Every experiment in this notebook is one or more *runs*, and every run is fully described by a `RunConfig`: which optimizer, which loss, which scene, which random seed, the learning rate, the iteration budget, and the model and rendering settings. A single function, `run_experiment`, consumes a `RunConfig` and does everything else — builds the model, optimizer, and loss; loads the scene (cached, so a resolution is decoded only once); runs the training loop; evaluates on the validation views at a fixed interval and on the test views at the end; and writes the entire history to a JSON file under `outputs/runs/`.

Logging to disk matters: the full comparison is on the order of a hundred runs, and an interrupted session must not lose the runs already completed. Each result file is named by a hash of its configuration, so re-running an identical configuration overwrites cleanly, and a finished run can be reloaded instead of recomputed.

Holding the harness fixed is what makes the comparison fair: optimizers, losses, and representations differ *only* in the component under study, because every other choice flows from the same `RunConfig` through the same code path — same data sampling, same initialisation, same seed and iteration budgets.


In [ ]:
@dataclass
class RunConfig:
    """Complete, hashable description of a single training run."""
    optimizer: str = "adam"        # sgd | momentum | nesterov | adam | adamw
    loss: str = "l2"               # key into LOSSES
    scene: str = "lego"
    seed: int = 0
    lr: float = 5e-4
    n_iterations: int = 40000
    batch_rays: int = 4096
    resolution: int = 200
    # model / rendering
    mlp_width: int = 128
    mlp_depth: int = 4
    n_freqs: int = 10
    n_samples: int = 64
    near: float = 2.0
    far: float = 6.0
    # optimizer extras
    momentum: float = 0.9          # used by momentum / nesterov
    weight_decay: float = 1e-2     # used by adamw
    use_schedule: bool = False
    warmup_steps: int = 1000
    # logging
    eval_every: int = 2000

    def run_id(self):
        """Deterministic short id from the configuration (used as filename)."""
        blob = json.dumps(asdict(self), sort_keys=True)
        digest = hashlib.sha1(blob.encode()).hexdigest()[:10]
        return f"{self.scene}_{self.optimizer}_{self.loss}_s{self.seed}_{digest}"


@dataclass
class RunResult:
    """Outcome of a run: the configuration plus the full logged history."""
    config: dict
    run_id: str
    loss_history: list      # list of [iter, loss]
    val_history: list       # list of [iter, psnr, ssim]
    test_metrics: dict      # {psnr, ssim, lpips} on the test views
    best_val_psnr: float
    wall_time_s: float
    iter_per_s: float


### Optimizer factory and scene cache

The harness selects an optimizer by name, building one of the five from-scratch classes of Section 4. Decoded scenes are cached by `(scene, resolution)`, so a resolution is read from disk and downsampled only once however many runs use it.


In [ ]:
def make_optimizer(name, params, lr, cfg):
    """Build one of the five from-scratch optimizers by name."""
    params = list(params)
    if name == "sgd":
        return MySGD(params, lr=lr)
    if name == "momentum":
        return MySGD(params, lr=lr, momentum=cfg.momentum)
    if name == "nesterov":
        return MySGD(params, lr=lr, momentum=cfg.momentum, nesterov=True)
    if name == "adam":
        return MyAdam(params, lr=lr)
    if name == "adamw":
        return MyAdamW(params, lr=lr, weight_decay=cfg.weight_decay)
    raise KeyError(f"unknown optimizer '{name}'")


# Scenes are decoded once per (scene, resolution) and reused across runs.
_scene_cache = {}


def load_scene_splits(scene, resolution):
    """Load and cache the train / val / test tensors for one scene."""
    key = (scene, resolution)
    if key not in _scene_cache:
        tr_i, tr_p, focal = load_scene(scene, "train", resolution)
        va_i, va_p, _ = load_scene(scene, "val", resolution, n_views=N_VAL_VIEWS)
        te_i, te_p, _ = load_scene(scene, "test", resolution, n_views=N_TEST_VIEWS)
        _scene_cache[key] = {
            "train": (tr_i.to(DEVICE), tr_p.to(DEVICE)),
            "val": (va_i.to(DEVICE), va_p.to(DEVICE)),
            "test": (te_i.to(DEVICE), te_p.to(DEVICE)),
            "focal": focal,
        }
    return _scene_cache[key]


### The training-and-evaluation run

`run_experiment` ties everything together: it builds the model, optimizer, and loss from the configuration, runs the stochastic training loop with periodic validation, evaluates on the test views at the end, writes the result to disk, and returns it. A configuration already on disk is reloaded instead of recomputed, so the experiment matrix can be built up incrementally and safely across sessions.


In [ ]:
RUNS_DIR = "../outputs/runs"


def run_experiment(cfg, with_lpips=True, save=True, reuse=True, verbose=True):
    """Train one NeRF under `cfg`, log the history, and return a RunResult.

    If `reuse` is set and a result file for this exact configuration already
    exists, it is loaded from disk instead of being recomputed."""
    os.makedirs(RUNS_DIR, exist_ok=True)
    run_id = cfg.run_id()
    path = os.path.join(RUNS_DIR, run_id + ".json")
    if reuse and os.path.exists(path):
        with open(path) as f:
            return RunResult(**json.load(f))

    seed_everything(cfg.seed)
    data = load_scene_splits(cfg.scene, cfg.resolution)
    train_i, train_p = data["train"]
    val_i, val_p = data["val"]
    test_i, test_p = data["test"]
    focal = data["focal"]
    H = W = cfg.resolution

    encoding = PositionalEncoding(num_freqs=cfg.n_freqs).to(DEVICE)
    model = TinyNeRF(encoding.out_dim, width=cfg.mlp_width,
                     depth=cfg.mlp_depth).to(DEVICE)
    opt = make_optimizer(cfg.optimizer, model.parameters(), cfg.lr, cfg)
    loss_fn = make_loss(cfg.loss)

    loss_history, val_history = [], []
    t0 = time.time()
    iterator = range(1, cfg.n_iterations + 1)
    if verbose:
        iterator = tqdm(iterator, desc=run_id)

    for it in iterator:
        if cfg.use_schedule:
            opt.lr = cosine_warmup_lr(it, cfg.lr, cfg.warmup_steps,
                                      cfg.n_iterations)

        idx = np.random.randint(len(train_i))
        rays_o, rays_d = get_rays(H, W, focal, train_p[idx])
        rays_o, rays_d = rays_o.reshape(-1, 3), rays_d.reshape(-1, 3)
        target = train_i[idx].reshape(-1, 3)

        pix = torch.randint(0, H * W, (cfg.batch_rays,), device=DEVICE)
        pred = render_rays(rays_o[pix], rays_d[pix], model, encoding,
                           near=cfg.near, far=cfg.far, N=cfg.n_samples)
        loss = loss_fn(pred, target[pix])

        opt.zero_grad()
        loss.backward()
        opt.step()

        if it == 1 or it % 200 == 0:
            loss_history.append([it, float(loss.item())])
        if it % cfg.eval_every == 0:
            m = evaluate(model, encoding, val_i, val_p, focal,
                         near=cfg.near, far=cfg.far, n_samples=cfg.n_samples)
            val_history.append([it, m["psnr"], m["ssim"]])
            if verbose:
                iterator.set_postfix(val_psnr=f"{m['psnr']:.2f}")

    wall = time.time() - t0
    test_metrics = evaluate(model, encoding, test_i, test_p, focal,
                            near=cfg.near, far=cfg.far, n_samples=cfg.n_samples,
                            with_lpips=with_lpips)
    best_val = max((p for _, p, _ in val_history), default=float("nan"))

    result = RunResult(
        config=asdict(cfg), run_id=run_id,
        loss_history=loss_history, val_history=val_history,
        test_metrics=test_metrics, best_val_psnr=best_val,
        wall_time_s=wall, iter_per_s=cfg.n_iterations / wall,
    )
    if save:
        with open(path, "w") as f:
            json.dump(asdict(result), f, indent=1)
    del model, encoding, opt
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return result


## 6.4 Harness Smoke Test

Before the full experiment matrix, a short run confirms the harness is wired correctly end to end: scene loading, model and optimizer construction, the training loop, periodic validation, the final test evaluation including LPIPS, and disk logging. The cell below runs 300 iterations (not part of the reported results) and prints the test metrics and throughput. A healthy smoke test shows the validation PSNR rising and finishes in a few seconds on the GPU.


In [ ]:
# Smoke test: a short run to confirm the harness wiring end to end.
# Not part of the reported experiments; uses a tiny iteration budget.
smoke_cfg = RunConfig(optimizer="adam", n_iterations=300, eval_every=150, seed=0)
smoke_result = run_experiment(smoke_cfg, with_lpips=True, save=False, verbose=True)

print("validation history:", smoke_result.val_history)
print("test metrics:      ", smoke_result.test_metrics)
print(f"throughput:         {smoke_result.iter_per_s:.1f} iter/s")


# 7. Optimizer Comparison

*To be completed in Stage 4.*

This section runs the five optimizers of Section 4 through the harness and compares them. Each optimizer is given its own learning rate, selected by a sweep on the validation views (plain SGD needs a far larger rate than Adam, so a shared rate would make it look broken rather than merely slow), and each configuration is repeated over at least three seeds. The comparison reports convergence speed, training stability, and held-out PSNR / SSIM / LPIPS, as `pandas` tables and as convergence plots overlaying the methods on log axes.


# 8. Loss Function Comparison

*To be completed in Stage 4.*

This section first introduces the two remaining loss formulations of Section 5 — SSIM and the weighted L1+SSIM combination — together with the patch-based ray sampling they require (SSIM is a spatial measure and cannot be computed from scattered pixels). It then compares all four losses under the best optimizer from Section 7, on the same scenes and metrics.


# 9. Proposed Improvements

*To be completed in Stage 6.*

Guided by the bottleneck the Section 7 comparison reveals, this section implements and evaluates the candidate improvements from the proposal — adaptive view sampling, multi-scale (coarse-to-fine) training, learning-rate restarts (SGDR), and a perceptual loss term. Each is measured as an isolated ablation against the best baseline configuration.


# 10. Gaussian Splatting

*To be completed in Stage 5.*

This section sets up an open-source 3D Gaussian Splatting reference implementation and trains it on the same scenes as NeRF, recording reconstruction quality, training time, and parameter count. Gaussian Splatting replaces the implicit MLP with an explicit set of optimised 3D Gaussians: an entirely different parameterisation of the same reconstruction problem.


# 11. NeRF vs Gaussian Splatting Comparison

*To be completed in Stage 5.*

This section contrasts the two scene representations head to head on the same held-out views and metrics: reconstruction quality (PSNR / SSIM / LPIPS), training time, and parameter count, placing the optimizer study of Sections 7 to 9 against an alternative way of formulating the reconstruction.


# 12. Conclusions and Future Work

*To be completed in Stage 7.*

This section synthesises the findings: which optimizer and loss converged fastest and most stably and gave the best held-out quality, how NeRF compared with Gaussian Splatting, and which proposed improvement helped most. It ties the results back to the problem formulation of Section 2 and outlines the directions left for future work.
